# Raw to Silver: Data Preprocessing
This notebook demonstrates how to preprocess transaction data from the raw layer and save a cleaned version to the silver layer. You can adapt this workflow for other raw datasets in your project.

In [9]:


import pandas as pd

RAW_PATH = "../raw-data/users_dataset.csv"
SILVER_PATH = "../Silver-data/users_clean.csv"

# Load raw users data
users = pd.read_csv(RAW_PATH)
print("Raw Users Shape:", users.shape)
users.head()

Raw Users Shape: (525, 7)


,user_id,name,email,phone,shipping_address,signup_ts,is_active
0,1.0,Christian Carter,abarnett@example.com,NaN,NaN,2024-06-08,True
1,2.0,Pedro Burton,daisy23@example.org,+1-347-816-2226,NaN,2024-08-23,False
2,3.0,Bryan Ross,jeffreysparks@example.org,713.856.0974x95,"23191 Tonya Court Suite 106, West Brandon, MI ...",2024-07-05,True
3,4.0,Sara Nunez,lisa81@example.com,+1-780-726-3058,"Unit 5107 Box 0248, DPO AE 86541",2022-10-22,True
4,5.0,Rachel Randall DDS,cobbshari@example.org,4088291592,"6103 Samuel Greens, Jamesfort, MH 13339",2022-08-09,True


## Step 1: Remove Duplicates

In [10]:
# Remove duplicate rows
users = users.drop_duplicates()
print("After removing duplicates:", users.shape)

After removing duplicates: (508, 7)


## Step 2: Handle Missing Values

In [24]:
# Check for missing values
missing_summary = users.isnull().sum()
print("Missing values per column:\n", missing_summary)


Missing values per column:
 user_id             0
name                0
email               0
phone               0
shipping_address    0
signup_ts           0
is_active           0
dtype: int64


In [17]:
#fill user Id 
users['user_id'] = range(1, len(users) + 1)

#fill eamil
users['email'] = users.apply(
    lambda r: r['email'] 
    if pd.notna(r['email']) 
    else f"user_{int(r['user_id'])}@unknown.com",
    axis=1
)

# Ensure email column is string
users['email'] = users['email'].astype(str)


In [19]:
# Fill missing names based on email
def name_from_email(email):
    base = email.split('@')[0]
    base = base.replace('.', ' ').replace('_', ' ')
    return base.title()

users['name'] = users.apply(
    lambda r: r['name'] if pd.notna(r['name']) else name_from_email(r['email']),
    axis=1
)


In [ ]:
#fill remaining missing values with defaults
users['phone'] = users['phone'].fillna('UNKNOWN_PHONE')
users['shipping_address'] = users['shipping_address'].fillna('ADDRESS_NOT_PROVIDED')
users['is_active'] = users['is_active'].fillna(True)


/var/folders/l_/c9wspj0n4453_nqxd1zd06d40000gn/T/ipykernel_43050/1131008855.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  users['is_active'] = users['is_active'].fillna(True)


In [23]:
users['signup_ts'] = pd.to_datetime(users['signup_ts'], errors='coerce')
users['signup_ts'] = users['signup_ts'].fillna(users['signup_ts'].median())


In [25]:
users.head()

,user_id,name,email,phone,shipping_address,signup_ts,is_active
0,1,Christian Carter,abarnett@example.com,UNKNOWN_PHONE,ADDRESS_NOT_PROVIDED,2024-06-08,True
1,2,Pedro Burton,daisy23@example.org,+1-347-816-2226,ADDRESS_NOT_PROVIDED,2024-08-23,False
2,3,Bryan Ross,jeffreysparks@example.org,713.856.0974x95,"23191 Tonya Court Suite 106, West Brandon, MI ...",2024-07-05,True
3,4,Sara Nunez,lisa81@example.com,+1-780-726-3058,"Unit 5107 Box 0248, DPO AE 86541",2022-10-22,True
4,5,Rachel Randall DDS,cobbshari@example.org,4088291592,"6103 Samuel Greens, Jamesfort, MH 13339",2022-08-09,True


## Step 3: Data Type Corrections (if needed)

In [26]:
users.dtypes

user_id                      int64
name                        object
email                       object
phone                       object
shipping_address            object
signup_ts           datetime64[ns]
is_active                     bool
dtype: object

## Step 4: Save Cleaned Data to Silver Layer

In [27]:
# Save the cleaned dataframe to the silver layer
import os
os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
users.to_csv(SILVER_PATH, index=False)
print(f"Cleaned transactions data saved to {SILVER_PATH}")

Cleaned transactions data saved to ../Silver-data/users_clean.csv


---

Repeat this process for each raw dataset (users, products, shops, etc.) by changing the input/output paths and adapting the cleaning steps as needed.